# 01: Data Quality, Schema Validation & Temporal Splitting
This notebook demonstrates loading raw transactional interactions, performing comprehensive data validation, checking sparsity, and applying a strict temporal split to prevent data leakage.

In [ ]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np

# Robust root resolver whether run from notebooks/ or workspace root
_curr = Path('.').resolve()
PROJECT_ROOT = _curr if (_curr / 'src').exists() else (_curr.parent if (_curr.parent / 'src').exists() else (_curr.parents[1] if len(_curr.parents) > 1 and (_curr.parents[1] / 'src').exists() else _curr))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Switch working directory to project root so all relative paths ('data/...', 'artifacts/...') resolve seamlessly
os.chdir(PROJECT_ROOT)

from src.data.loader import DataLoader
from src.data.validator import DataValidator
from src.data.preprocess import DataPreprocessor


## 1. Load Raw Transactions and Product Catalog

In [ ]:
loader = DataLoader('data/raw/retail_events.csv', 'data/raw/product_catalog.csv')
events_df, catalog_df = loader.load_and_merge()
print(f"Loaded {len(events_df):,} events across {events_df['customer_id'].nunique():,} customers and {events_df['product_id'].nunique():,} products.")
events_df.head()


## 2. Execute Data Validation & Quality Checks

In [ ]:
validator = DataValidator(output_report_path='outputs/data_quality_report.json')
clean_df, dq_report = validator.validate_and_clean(events_df)
print("Data Quality Report Summary:")
print(f"- Validated Rows: {dq_report['row_count']:,}")
print(f"- Interaction Sparsity: {dq_report['sparsity']*100:.3f}%")
print(f"- Event Distribution: {dq_report['event_distribution']}")


## 3. Strict Temporal Train / Validation / Test Split

In [ ]:
preprocessor = DataPreprocessor(train_ratio=0.70, val_ratio=0.15, test_ratio=0.15)
train_df, val_df, test_df, cutoffs = preprocessor.temporal_split(clean_df)
print(f"Train events: {len(train_df):,} ({train_df['timestamp'].min()} to {cutoffs['train_val_cutoff']})")
print(f"Val events  : {len(val_df):,} ({cutoffs['train_val_cutoff']} to {cutoffs['val_test_cutoff']})")
print(f"Test events : {len(test_df):,} ({cutoffs['val_test_cutoff']} to {test_df['timestamp'].max()})")
